In [ ]:
def main(datasources, start_date, end_date):
    """AI02：日内八段微观结构形状对下一日特异收益的集成预测。"""
    import gc
    import time
    import numpy as np
    import pandas as pd
    import dai
    import structlog
    import xgboost as xgb

    logger = structlog.get_logger()

    # 正式候选版：2019—2024训练；平台传入的2025/2026区间只用于预测。
    TRAIN_START = "2019-01-01 00:00:00"
    TRAIN_END = "2024-12-31 23:59:59"
    RECENT_START = pd.Timestamp("2023-01-01")
    TRAIN_BAR_TABLE = "bigalpha_2026_stock_bar1m"
    LOOKBACK_DAYS = 60

    profile_metrics = [
        "return",
        "amount_share",
        "imbalance",
        "order_imbalance",
        "ofi",
        "micro_deviation",
        "spread",
        "volatility",
    ]
    profile_feature_columns = [
        f"{metric}_b{bucket}"
        for metric in profile_metrics
        for bucket in range(8)
    ]
    daily_feature_columns = [
        "daily_return",
        "close_location",
        "realized_volatility",
        "realized_skewness",
        "price_efficiency",
        "quote_validity",
        "auction_return",
        "auction_amount_share",
        "ofi_price_alignment",
        "book_absorption",
        "log_amount",
    ]
    raw_feature_columns = profile_feature_columns + daily_feature_columns
    model_feature_columns = [f"rank_{column}" for column in raw_feature_columns]
    nuisance_columns = [
        "daily_return",
        "return_mean_5d",
        "return_mean_20d",
        "return_std_20d",
        "log_amount",
    ]

    def query_profile_features(bar_table, sd, ed):
        query_start = pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)

        bucket_pivots = []
        profile_selects = []
        for bucket in range(8):
            bucket_pivots.extend(
                [
                    f"MAX(CASE WHEN bucket_id={bucket} THEN bucket_return END) AS return_b{bucket}",
                    f"MAX(CASE WHEN bucket_id={bucket} THEN bucket_amount END) AS bucket_amount_b{bucket}",
                    f"MAX(CASE WHEN bucket_id={bucket} THEN bucket_imbalance END) AS imbalance_b{bucket}",
                    f"MAX(CASE WHEN bucket_id={bucket} THEN bucket_order_imbalance END) AS order_imbalance_b{bucket}",
                    f"MAX(CASE WHEN bucket_id={bucket} THEN bucket_ofi END) AS ofi_b{bucket}",
                    f"MAX(CASE WHEN bucket_id={bucket} THEN bucket_micro_deviation END) AS micro_deviation_b{bucket}",
                    f"MAX(CASE WHEN bucket_id={bucket} THEN bucket_spread END) AS spread_b{bucket}",
                    f"MAX(CASE WHEN bucket_id={bucket} THEN bucket_volatility END) AS volatility_b{bucket}",
                ]
            )
            profile_selects.extend(
                [
                    f"profile.return_b{bucket}",
                    f"CASE WHEN daily.daily_amount > 0 THEN profile.bucket_amount_b{bucket} / daily.daily_amount ELSE NULL END AS amount_share_b{bucket}",
                    f"profile.imbalance_b{bucket}",
                    f"profile.order_imbalance_b{bucket}",
                    f"profile.ofi_b{bucket}",
                    f"profile.micro_deviation_b{bucket}",
                    f"profile.spread_b{bucket}",
                    f"profile.volatility_b{bucket}",
                ]
            )
        bucket_pivot_sql = ",\n                ".join(bucket_pivots)
        profile_select_sql = ",\n                ".join(profile_selects)
        output_column_sql = ",\n            ".join(raw_feature_columns + nuisance_columns[1:])

        sql = f"""
        WITH raw AS (
            SELECT
                date,
                instrument,
                CAST(date_trunc('day', date) AS DATE) AS trading_day,
                CAST(strftime(date, '%H%M') AS INTEGER) AS hhmm,
                CASE
                    WHEN CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 931 AND 1000 THEN 0
                    WHEN CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 1001 AND 1030 THEN 1
                    WHEN CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 1031 AND 1100 THEN 2
                    WHEN CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 1101 AND 1130 THEN 3
                    WHEN CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 1300 AND 1329 THEN 4
                    WHEN CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 1330 AND 1359 THEN 5
                    WHEN CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 1400 AND 1429 THEN 6
                    WHEN CAST(strftime(date, '%H%M') AS INTEGER) BETWEEN 1430 AND 1457 THEN 7
                    ELSE NULL
                END AS bucket_id,
                CAST(pre_close AS DOUBLE) AS pre_close,
                CAST(high AS DOUBLE) AS high,
                CAST(low AS DOUBLE) AS low,
                CAST(close AS DOUBLE) AS close,
                CAST(amount AS DOUBLE) AS amount,
                CASE
                    WHEN bid_price1 > 0 AND ask_price1 > 0 AND ask_price1 >= bid_price1
                    THEN 1 ELSE 0
                END AS valid_quote,
                CASE
                    WHEN bid_price1 > 0 AND ask_price1 > 0 AND ask_price1 >= bid_price1
                    THEN (bid_price1 + ask_price1) / 2.0 ELSE NULL
                END AS mid_price,
                CASE
                    WHEN bid_price1 > 0 AND ask_price1 > 0 AND ask_price1 >= bid_price1
                    THEN (ask_price1 - bid_price1) / ((ask_price1 + bid_price1) / 2.0)
                    ELSE NULL
                END AS relative_spread,
                COALESCE(bid_volume1, 0) * 1.0
                  + COALESCE(bid_volume2, 0) * EXP(-0.35)
                  + COALESCE(bid_volume3, 0) * EXP(-0.70) AS weighted_bid,
                COALESCE(ask_volume1, 0) * 1.0
                  + COALESCE(ask_volume2, 0) * EXP(-0.35)
                  + COALESCE(ask_volume3, 0) * EXP(-0.70) AS weighted_ask,
                COALESCE(bid_num_orders1, 0) + COALESCE(bid_num_orders2, 0)
                  + COALESCE(bid_num_orders3, 0) AS bid_orders,
                COALESCE(ask_num_orders1, 0) + COALESCE(ask_num_orders2, 0)
                  + COALESCE(ask_num_orders3, 0) AS ask_orders,
                CASE
                    WHEN bid_price1 > 0 AND ask_price1 > 0
                     AND COALESCE(bid_volume1, 0) + COALESCE(ask_volume1, 0) > 0
                    THEN (
                        ask_price1 * COALESCE(bid_volume1, 0)
                        + bid_price1 * COALESCE(ask_volume1, 0)
                    ) / (COALESCE(bid_volume1, 0) + COALESCE(ask_volume1, 0))
                    ELSE NULL
                END AS micro_price
            FROM {{bar_table}}
        ),
        lagged AS (
            SELECT
                *,
                LAG(close) OVER (
                    PARTITION BY trading_day, instrument ORDER BY date
                ) AS previous_close,
                LAG(weighted_bid) OVER (
                    PARTITION BY trading_day, instrument ORDER BY date
                ) AS previous_weighted_bid,
                LAG(weighted_ask) OVER (
                    PARTITION BY trading_day, instrument ORDER BY date
                ) AS previous_weighted_ask
            FROM raw
        ),
        minute_features AS (
            SELECT
                *,
                CASE WHEN close > 0 AND previous_close > 0
                    THEN LN(close / previous_close) ELSE NULL END AS minute_return,
                CASE WHEN valid_quote = 1 AND weighted_bid + weighted_ask > 0
                    THEN (weighted_bid - weighted_ask)
                       / (weighted_bid + weighted_ask + 1e-12) ELSE NULL END
                    AS l3_imbalance,
                CASE WHEN valid_quote = 1 AND bid_orders + ask_orders > 0
                    THEN (bid_orders - ask_orders) * 1.0
                       / (bid_orders + ask_orders + 1e-12) ELSE NULL END
                    AS order_imbalance,
                CASE
                    WHEN valid_quote = 1
                     AND previous_weighted_bid IS NOT NULL
                     AND previous_weighted_ask IS NOT NULL
                    THEN (
                        (weighted_bid - previous_weighted_bid)
                        - (weighted_ask - previous_weighted_ask)
                    ) / (
                        weighted_bid + weighted_ask
                        + previous_weighted_bid + previous_weighted_ask + 1e-12
                    ) ELSE NULL
                END AS ofi,
                CASE WHEN valid_quote = 1 AND mid_price > 0 AND micro_price IS NOT NULL
                    THEN (micro_price - mid_price) / mid_price ELSE NULL END
                    AS micro_deviation
            FROM lagged
        ),
        bucket_stats AS (
            SELECT
                trading_day,
                instrument,
                bucket_id,
                SUM(minute_return) AS bucket_return,
                SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END) AS bucket_amount,
                AVG(l3_imbalance) AS bucket_imbalance,
                AVG(order_imbalance) AS bucket_order_imbalance,
                AVG(ofi) AS bucket_ofi,
                AVG(micro_deviation) AS bucket_micro_deviation,
                AVG(relative_spread) AS bucket_spread,
                SQRT(SUM(minute_return * minute_return)) AS bucket_volatility
            FROM minute_features
            WHERE bucket_id IS NOT NULL
            GROUP BY trading_day, instrument, bucket_id
        ),
        bucket_profile AS (
            SELECT
                trading_day,
                instrument,
                {{bucket_pivot_sql}}
            FROM bucket_stats
            GROUP BY trading_day, instrument
        ),
        daily_stats AS (
            SELECT
                trading_day,
                instrument,
                ARG_MIN(pre_close, date) FILTER (WHERE pre_close > 0) AS pre_close_px,
                ARG_MIN(close, date) FILTER (WHERE close > 0) AS first_price,
                ARG_MAX(close, date) FILTER (WHERE close > 0) AS close_price,
                MAX(high) FILTER (WHERE high > 0) AS high_price,
                MIN(low) FILTER (WHERE low > 0) AS low_price,
                ARG_MAX(close, date) FILTER (
                    WHERE hhmm BETWEEN 1300 AND 1457 AND close > 0
                ) AS price_1457,
                ARG_MAX(close, date) FILTER (WHERE hhmm = 1500 AND close > 0)
                    AS price_1500,
                SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END) AS daily_amount,
                SUM(CASE WHEN hhmm = 1500 AND amount > 0 THEN amount ELSE 0 END)
                    AS auction_amount,
                SUM(minute_return * minute_return) AS sum_squared_return,
                SUM(minute_return * minute_return * minute_return) AS sum_cubed_return,
                SUM(ABS(minute_return)) AS sum_absolute_return,
                SUM(ofi * minute_return) AS ofi_return_product,
                SUM(ofi * ofi) AS sum_squared_ofi,
                AVG(valid_quote) AS quote_validity,
                AVG(l3_imbalance) AS mean_imbalance
            FROM minute_features
            GROUP BY trading_day, instrument
        ),
        daily_calculated AS (
            SELECT
                *,
                CASE WHEN close_price > 0 AND pre_close_px > 0
                    THEN close_price / pre_close_px - 1 ELSE NULL END AS daily_return,
                CASE WHEN high_price > low_price
                    THEN (close_price - low_price) / (high_price - low_price)
                    ELSE 0.5 END AS close_location,
                SQRT(COALESCE(sum_squared_return, 0)) AS realized_volatility,
                CASE WHEN sum_squared_return > 1e-12
                    THEN sum_cubed_return / (POWER(sum_squared_return, 1.5) + 1e-12)
                    ELSE 0 END AS realized_skewness,
                CASE WHEN sum_absolute_return > 1e-12 AND close_price > 0 AND first_price > 0
                    THEN ABS(LN(close_price / first_price)) / sum_absolute_return
                    ELSE 0 END AS price_efficiency,
                CASE WHEN price_1500 > 0 AND price_1457 > 0
                    THEN LN(price_1500 / price_1457) ELSE NULL END AS auction_return,
                CASE WHEN daily_amount > 0 THEN auction_amount / daily_amount ELSE NULL END
                    AS auction_amount_share,
                CASE WHEN sum_squared_ofi > 1e-12 AND sum_squared_return > 1e-12
                    THEN ofi_return_product
                       / (SQRT(sum_squared_ofi * sum_squared_return) + 1e-12)
                    ELSE 0 END AS ofi_price_alignment,
                CASE WHEN close_price > 0 AND pre_close_px > 0
                    THEN -mean_imbalance * (close_price / pre_close_px - 1)
                    ELSE NULL END AS book_absorption,
                CASE WHEN daily_amount > 0 THEN LN(daily_amount) ELSE NULL END AS log_amount
            FROM daily_stats
        ),
        final_base AS (
            SELECT
                daily.trading_day,
                daily.instrument,
                {{profile_select_sql}},
                daily.daily_return,
                daily.close_location,
                daily.realized_volatility,
                daily.realized_skewness,
                daily.price_efficiency,
                daily.quote_validity,
                daily.auction_return,
                daily.auction_amount_share,
                daily.ofi_price_alignment,
                daily.book_absorption,
                daily.log_amount
            FROM daily_calculated AS daily
            LEFT JOIN bucket_profile AS profile
              ON daily.trading_day = profile.trading_day
             AND daily.instrument = profile.instrument
        ),
        daily_rolling AS (
            SELECT
                *,
                AVG(daily_return) OVER (
                    PARTITION BY instrument ORDER BY trading_day
                    ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
                ) AS return_mean_5d,
                AVG(daily_return) OVER (
                    PARTITION BY instrument ORDER BY trading_day
                    ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
                ) AS return_mean_20d,
                STDDEV_SAMP(daily_return) OVER (
                    PARTITION BY instrument ORDER BY trading_day
                    ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
                ) AS return_std_20d
            FROM final_base
        )
        SELECT
            CAST(trading_day AS DATETIME) AS date,
            instrument,
            {{output_column_sql}}
        FROM daily_rolling
        ORDER BY date, instrument
        """.format(
            bar_table=bar_table,
            bucket_pivot_sql=bucket_pivot_sql,
            profile_select_sql=profile_select_sql,
            output_column_sql=output_column_sql,
        )
        frame = dai.query(
            sql,
            filters={"date": [query_start, ed]},
            compression=True,
        ).df()
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        frame["instrument"] = frame["instrument"].astype(str)
        return frame.sort_values(["instrument", "date"]).reset_index(drop=True)

    def residualize_training_labels(data):
        label_output = np.full(len(data), np.nan, dtype="float32")
        nuisance_rank_columns = [f"rank_{column}" for column in nuisance_columns]
        for _, row_index in data.groupby("date", sort=False).groups.items():
            positions = np.asarray(row_index, dtype=int)
            y = data.loc[positions, "label_rank"].to_numpy(dtype="float64")
            valid = np.isfinite(y)
            if valid.sum() < 100:
                continue
            x = data.loc[positions, nuisance_rank_columns].to_numpy(dtype="float64")
            x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
            design = np.column_stack([np.ones(len(x)), x])
            design_valid = design[valid]
            y_valid = y[valid]
            penalty = np.eye(design_valid.shape[1]) * 5.0
            penalty[0, 0] = 0.0
            beta = np.linalg.solve(
                design_valid.T @ design_valid + penalty,
                design_valid.T @ y_valid,
            )
            residual = y_valid - design_valid @ beta
            residual = residual - residual.mean()
            scale = residual.std(ddof=0)
            if scale > 1e-12:
                residual = residual / scale
            residual = np.clip(residual, -5.0, 5.0)
            label_output[positions[valid]] = residual.astype("float32")
        return label_output

    def prepare_dataset(bar_table, sd, ed, include_label):
        started = time.time()
        price = query_profile_features(bar_table, sd, ed)
        numeric_columns = [
            column for column in price.columns if column not in {"date", "instrument"}
        ]
        for column in numeric_columns:
            price[column] = pd.to_numeric(price[column], errors="coerce")
        price[numeric_columns] = price[numeric_columns].replace(
            [np.inf, -np.inf], np.nan
        )

        if include_label:
            price["label_raw"] = price.groupby("instrument")["daily_return"].shift(-1)

        target_start = pd.to_datetime(sd).normalize()
        target_end = pd.to_datetime(ed).normalize()
        price = price[(price["date"] >= target_start) & (price["date"] <= target_end)]

        stock_pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]},
        ).df()
        stock_pool["date"] = pd.to_datetime(stock_pool["date"]).dt.normalize()
        stock_pool["instrument"] = stock_pool["instrument"].astype(str)
        data = pd.merge(stock_pool, price, how="left", on=["date", "instrument"])

        rank_source_columns = list(dict.fromkeys(raw_feature_columns + nuisance_columns))
        for column in rank_source_columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")
            data[column] = data[column].replace([np.inf, -np.inf], np.nan)
            data[f"rank_{column}"] = (
                data.groupby("date")[column].rank(method="average", pct=True) - 0.5
            ).astype("float32")

        if include_label:
            data["label_rank"] = (
                data.groupby("date")["label_raw"].rank(method="average", pct=True) - 0.5
            )
            data["label"] = residualize_training_labels(data)

        keep_columns = ["date", "instrument"] + model_feature_columns
        if include_label:
            keep_columns.append("label")
        output = data[keep_columns].sort_values(["date", "instrument"]).reset_index(drop=True)
        logger.info(
            "AI02特征集完成",
            start=str(sd),
            end=str(ed),
            rows=len(output),
            features=len(model_feature_columns),
            elapsed=round(time.time() - started, 2),
        )
        return output

    def new_model(n_estimators, random_state):
        return xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=n_estimators,
            max_depth=3,
            learning_rate=0.04,
            min_child_weight=100,
            subsample=0.80,
            colsample_bytree=0.70,
            reg_alpha=0.30,
            reg_lambda=8.0,
            max_bin=256,
            tree_method="hist",
            n_jobs=-1,
            random_state=random_state,
            verbosity=0,
        )

    logger.info("AI02构建训练集", train_start=TRAIN_START, train_end=TRAIN_END)
    train = prepare_dataset(
        TRAIN_BAR_TABLE,
        TRAIN_START,
        TRAIN_END,
        include_label=True,
    )
    train = train.dropna(subset=["label"]).reset_index(drop=True)
    train_matrix = train[model_feature_columns].fillna(0.0).to_numpy(dtype="float32")
    train_target = train["label"].to_numpy(dtype="float32")
    latest_training_date = train["date"].max()
    age_days = (latest_training_date - train["date"]).dt.days.to_numpy(dtype="float32")
    full_weight = np.power(0.5, age_days / 730.0).astype("float32")
    recent_mask = (train["date"] >= RECENT_START).to_numpy()

    logger.info(
        "AI02训练全样本模型",
        samples=len(train),
        features=len(model_feature_columns),
    )
    full_model = new_model(n_estimators=220, random_state=20260720)
    full_model.fit(train_matrix, train_target, sample_weight=full_weight)

    logger.info("AI02训练近期模型", samples=int(recent_mask.sum()))
    recent_model = new_model(n_estimators=180, random_state=20260721)
    recent_model.fit(
        train_matrix[recent_mask],
        train_target[recent_mask],
        sample_weight=full_weight[recent_mask],
    )

    del train_matrix, train_target, full_weight, age_days, train
    gc.collect()

    logger.info("AI02构建测试集", start=str(start_date), end=str(end_date))
    test = prepare_dataset(
        datasources["bar1m"],
        start_date,
        end_date,
        include_label=False,
    )
    test_matrix = test[model_feature_columns].fillna(0.0).to_numpy(dtype="float32")
    test["prediction_full"] = full_model.predict(test_matrix)
    test["prediction_recent"] = recent_model.predict(test_matrix)
    ranked_full = (
        test.groupby("date")["prediction_full"].rank(method="average", pct=True) - 0.5
    )
    ranked_recent = (
        test.groupby("date")["prediction_recent"].rank(method="average", pct=True) - 0.5
    )
    test["factor"] = 0.65 * ranked_full + 0.35 * ranked_recent
    test["factor"] = pd.to_numeric(test["factor"], errors="coerce")
    test["factor"] = test["factor"].replace([np.inf, -np.inf], np.nan)

    result = test[["date", "instrument", "factor"]].copy()
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    logger.info("AI02完成", rows=len(result), missing=int(result["factor"].isna().sum()))
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)
